# Environment / Dependencies Setup
The cell below installs the OCR stack, embedding/search tools, LangChain + OpenRouter, and dotenv.

In [ ]:
!pip install -q rapidocr onnxruntime pymupdf Pillow sentence-transformers "torch>=2.5" faiss-cpu langchain-openai requests numpy python-dotenv

### Getting an OpenRouter API key
One key works across many models — save it in `.env` as `OPENROUTER_API_KEY=...`.

In [ ]:
import logging
import os

import faiss
import fitz  # PyMuPDF
import numpy as np
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from PIL import Image
from rapidocr import RapidOCR
from sentence_transformers import SentenceTransformer

# Keep the output readable: hide model-download progress bars and noisy library logs
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for noisy in ("sentence_transformers", "huggingface_hub", "httpx", "transformers"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

### Setup: Load the Key and Create the Three Tools
The LLM, the embedding model, and the OCR engine, reused across Parts 1 and 2.

In [ ]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

# 1. The LLM: a free model on OpenRouter, so running this lab costs nothing
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=500,
    extra_body={"reasoning": {"enabled": False}},
)

# 2. The embedding model: runs on your CPU, no API key needed
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 3. The OCR engine: loads its models once; "error" hides the startup chatter
ocr_engine = RapidOCR(params={"Global.log_level": "error"})

print("Key loaded. LLM, embedding model and OCR engine are ready.")

`temperature=0` keeps answers predictable; `base_url` points at OpenRouter; hidden reasoning is disabled.

You should see: `Key loaded. ...ready.`

In [ ]:
def download(url, path):
    """Download a file from a URL and save it at the given path."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    response = requests.get(url)
    response.raise_for_status()
    with open(path, "wb") as f:
        f.write(response.content)
    print(f"Downloaded {os.path.basename(path)} ({len(response.content)/1024:.1f} KB)")
    return path

INVOICE_URLS = {
    "simple-invoice.png": "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/simple-invoice.png",
    "contoso-receipt.png": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/receipt/contoso-receipt.png",
    "purchase-order-1.jpg": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/forms/Form_1.jpg",
}

invoice_paths = {}
for filename, url in INVOICE_URLS.items():
    invoice_paths[filename] = download(url, os.path.join("data", "invoices", filename))

### Step 1.1: Download the Sample Documents
A small download helper, reused in Part 2.

In [ ]:
def read_text(image, keep_layout=True):
    """OCR one image (a file path or a numpy array) and return its text."""
    result = ocr_engine(image)
    if keep_layout:
        return result.to_markdown()      # rebuilds rows and columns from the box positions
    return "\n".join(result.txts)        # plain lines, in the order the engine found them

# document_texts maps each filename to its OCR text
document_texts = {}
for filename, path in invoice_paths.items():
    document_texts[filename] = read_text(path)
    print(f"{filename}: {len(document_texts[filename])} characters of text")

print()
print(document_texts["simple-invoice.png"])

### Step 1.2: OCR Each Document (Keeping the Layout)
`read_text` OCRs one image and returns layout-preserving text.

In [ ]:
flat_text = " ".join(ocr_engine(invoice_paths["simple-invoice.png"]).txts)
print(flat_text)

Flat text mixes every label and value onto one line. Only the layout version keeps columns aligned so an LLM can match each number to its header.

In [ ]:
def embed(texts):
    """Turn a list of texts into a 2D array of unit-length vectors, one row per text."""
    # unit-length vectors, so a dot product later equals cosine similarity
    return embedder.encode(texts, normalize_embeddings=True)

doc_names = list(document_texts)
doc_vectors = embed([document_texts[name] for name in doc_names])

print(f"Knowledge base built with {len(doc_names)} documents.")
print(f"Vector table shape: {doc_vectors.shape} (documents x dimensions)")

### Step 1.3: Embed Each Document (One Vector Each)
Short document -> one whole-document vector via the reusable `embed` helper.

In [ ]:
def search_documents(query, top_k=2):
    """Return the top_k closest documents as {source, text, score}, best first."""
    scores = doc_vectors @ embed([query])[0]          # one cosine score per document
    best = np.argsort(scores)[::-1][:top_k]           # positions of the highest scores
    return [
        {"source": doc_names[i], "text": document_texts[doc_names[i]], "score": float(scores[i])}
        for i in best
    ]

`normalize_embeddings=True` scales vectors to length 1. Every search returns the shared `{source, text, score}` shape.

In [ ]:
def ask_llm(query, hits):
    """Send the retrieved sources to the LLM and return its tagged answer."""
    # tag each block with its source so the model can copy the tag into its answer
    context = "\n\n".join(f"[Source: {hit['source']}]\n{hit['text']}" for hit in hits)
    prompt = f"""You answer questions using ONLY the context below.
Every fact you state must be followed by its source tag, copied exactly, like [Source: name].
If the answer is not in the context, say "Not found in the provided documents."

Context:
{context}

Question: {query}
Answer:"""
    return llm.invoke(prompt).content.strip()

def show_sources(answer, hits):
    """Explainability: for every retrieved source, was its tag cited in the answer?"""
    print("--- EXPLAINABILITY ---")
    for hit in hits:
        tag = f"[Source: {hit['source']}]"
        used = tag in answer
        print(f"{hit['score']:.3f}  {hit['source']}  ->  {'USED' if used else 'retrieved but NOT used'}")
        if used:
            for line in answer.splitlines():
                if tag in line:
                    print(f"       cited in: {line.strip()}")

def answer_question(query, hits):
    """Print the retrieved sources, the answer, and the explainability check."""
    print(f"Question: {query}\n")
    print("--- SOURCES RETRIEVED ---")
    for hit in hits:
        print(f"{hit['score']:.3f}  {hit['source']}")
    answer = ask_llm(query, hits)
    print("\n--- ANSWER ---")
    print(answer)
    print()
    show_sources(answer, hits)

### Step 1.5: Ask the LLM and Check the Sources
`ask_llm` tags every fact with its source; `show_sources` checks which tags appear in the answer.

In [ ]:
query = "What is the invoice number and total charge on the Contoso invoice?"
answer_question(query, search_documents(query))

### Step 1.6: Ask Questions
You should see `simple-invoice.png` ranked first and its figures cited with a `USED` tag.

In [ ]:
follow_up = "Which items were purchased on the Surface Pro receipt?"
answer_question(follow_up, search_documents(follow_up, top_k=3))

A different question should move a different document to the top — that is how you know retrieval is telling your documents apart.

In [ ]:
PDF_PATH = download(
    "https://raw.githubusercontent.com/jamalmazrui/pdf2ocr/master/debate.pdf",
    os.path.join("data", "debate.pdf"),
)

def render_pages(pdf_path, dpi=200):
    """Draw every PDF page as an image array. A higher dpi means a sharper image for OCR."""
    zoom = dpi / 72                                   # PDF pages are 72 dpi by default
    matrix = fitz.Matrix(zoom, zoom)
    pages = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=matrix)
            # raw page pixels -> RGB image -> numpy array for the OCR engine
            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pages.append(np.array(image))
    return pages

page_images = render_pages(PDF_PATH)
print(f"Rendered {len(page_images)} pages. Page 1 is {page_images[0].shape[1]}x{page_images[0].shape[0]} pixels.")

### Step 2.1: Download the PDF and Draw Each Page as an Image
OCR reads images, so every PDF page is rendered to an image first.

In [ ]:
page_texts = []
for page_number, image in enumerate(page_images, start=1):
    page_texts.append(read_text(image, keep_layout=False))
    if page_number <= 3:
        print(f"Page {page_number} transcribed: {len(page_texts[-1])} characters")

print(f"... all {len(page_texts)} pages done, {sum(len(t) for t in page_texts)} characters in total.")
print()
print(page_texts[3][:350])

OCR every page with the same `read_text`, now without layout (`keep_layout=False`) since this is flowing prose. Takes about a minute on a CPU.

In [ ]:
CHUNK_SIZE = 500
OVERLAP = 50
# step by CHUNK_SIZE - OVERLAP so consecutive chunks share OVERLAP characters
STEP = CHUNK_SIZE - OVERLAP

chunks = []
for page_number, text in enumerate(page_texts, start=1):
    for start in range(0, len(text), STEP):
        piece = text[start : start + CHUNK_SIZE].strip()
        if piece:
            chunks.append({
                "source": f"debate.pdf, page {page_number}, chunk {len(chunks) + 1}",
                "page": page_number,
                "text": piece,
            })

print(f"Total chunks created: {len(chunks)}")
print("End of chunk 1:  ", repr(chunks[0]["text"][-50:]))
print("Start of chunk 2:", repr(chunks[1]["text"][:50]))
print("Chunk 2 starts with text already inside chunk 1:", chunks[1]["text"][:40] in chunks[0]["text"])

### Step 2.3: Cut the Text into Overlapping Chunks
500-character chunks that overlap by 50, each labelled with its page and chunk number.

In [ ]:
chunk_vectors = embed([chunk["text"] for chunk in chunks]).astype("float32")  # FAISS needs float32

index = faiss.IndexFlatIP(chunk_vectors.shape[1])   # IP = inner product (dot product)
index.add(chunk_vectors)

print(f"Indexed {index.ntotal} chunks, {chunk_vectors.shape[1]} dimensions each.")

Normalized vectors make FAISS's inner product exactly Part 1's cosine score. You should see `Indexed ... chunks, 384 dimensions each`.

In [ ]:
def search_chunks(query, top_k=3):
    """Return the top_k closest chunks as {source, text, score}, best first."""
    query_vector = embed([query]).astype("float32")
    # faiss returns the rows already sorted by score, best first
    scores, positions = index.search(query_vector, top_k)
    return [
        {**chunks[i], "score": float(score)}
        for score, i in zip(scores[0], positions[0])
    ]

The results share Part 1's `{source, text, score}` shape, so the same display code works unchanged.

In [ ]:
query = "How long does each debate last, and are opening statements allowed?"
answer_question(query, search_chunks(query))

### Step 2.6: Ask Questions With an Explainable Answer
You should see three chunks retrieved and the explainability list.

In [ ]:
query = "What are the rules about props, notes and charts during a debate?"
answer_question(query, search_chunks(query, top_k=4))

An answer spread over several chunks needs several retrieved at once — the explainability check shows exactly which ones the answer drew from.

In [ ]:
def count_tokens(text):
    """How many word pieces ('tokens') the embedding model needs for this text."""
    return len(embedder.tokenizer.tokenize(text))

print(f"The embedding model reads at most {embedder.max_seq_length} tokens per text.\n")

print("Part 1 units (whole documents):")
for name, text in document_texts.items():
    print(f"  {name}: {len(text)} chars, {count_tokens(text)} tokens")

chunk_tokens = [count_tokens(chunk["text"]) for chunk in chunks]
print(f"\nPart 2 units (chunks): {len(chunks)} chunks, longest = {max(chunk_tokens)} tokens")

whole_pdf = "\n".join(page_texts)
# how much of the PDF falls outside the model window and is never embedded
print(f"\nThe whole PDF as ONE unit would be {count_tokens(whole_pdf)} tokens, "
      f"so about {100 - 100 * embedder.max_seq_length // count_tokens(whole_pdf)}% of it would never be seen by the embedding model.")

### Part 3: Whole Documents or Chunks?
Compares both retrieval units against the embedding model's length limit to explain the design choice.